In [1]:
import load_data as jtd
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs

In [2]:
df_library = pd.read_csv('algebrakit_firstbatch.csv')
collection_attributes = ['skillName', 'skillsTodo', 'stepType']

for colatt in collection_attributes:
    df_library[colatt] = df_library[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_library = jtd.additional_columns(df_library)

df_library.head()

,exerciseID,sessionID,sessionSuccess,creationTimestamp,finished,marksEarned,marksTotal,skillName,stepType,errors,...,errorDesc_DivisionMultiplication_count,errorDesc_MultiplyAllTerms_count,errorDesc_SignError_count,errorDesc_SkippingTerms_count,errorDesc_ChangingTermType_count,errorDesc_AddedDenominator_count,errorDesc_MultiplicationAddition_count,errorTypes_MISCONCEPTION_count,errorTypes_MISTAKE_count,interactionStepsNeeded
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,2
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,0
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4


In [3]:
eventflows_dict = {}

for ex in list(df_library['exerciseID'].unique()):
    ex_dict = {}
    df_ex = df_library[ df_library['exerciseID'] == ex]
        
    for sesh in list(df_ex['sessionID'].unique()):
        events = list(df_ex[ df_ex['sessionID'] == sesh]['eventType'].dropna())
        ex_dict[sesh] = events
    
    eventflows_dict[ex] = ex_dict


In [4]:
exercises = list(df_library['exerciseID'].unique())
exercise = exercises[7]

event_flows = eventflows_dict[exercise]

In [5]:
import pandas as pd
import plotly.express as px


# --- Convert to tidy DataFrame ---
rows = []
for sid, flow in event_flows.items():
    for step, event in enumerate(flow):
        rows.append({"session": sid, "step": step, "event": event})
df = pd.DataFrame(rows)

# --- Timeline-style line plot ---
fig = px.line(
    df, x="step", y="event", color="session",
    markers=True, line_shape="linear"
)
fig.update_layout(
    title="Timeline of All Event Flows",
    xaxis_title="Step",
    yaxis_title="Event Type",
    yaxis=dict(type="category", categoryorder="array",
               categoryarray=["EVALUATE", "HINT", "GIVEUP"])  # consistent event order
)
fig.show()



In [6]:
import pandas as pd
import plotly.graph_objects as go
from collections import Counter


# --- Count duplicate flows ---
flows_as_tuples = {sid: tuple(flow) for sid, flow in event_flows.items()}
flow_counts = Counter(flows_as_tuples.values())

# --- Prepare line segments per unique flow ---
fig = go.Figure()

for flow, count in flow_counts.items():
    x = list(range(len(flow)))
    y = list(flow)
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(width=1 + 2*(count-1)),  # base width 1, fatten for duplicates
        marker=dict(size=6),
        name=f"x{count}" if count>1 else None,  # optional legend for duplicates
        hovertemplate=f"Flow: {flow}<br>Count: {count}<extra></extra>"
    ))

fig.update_layout(
    title="Timeline of Event Flows (Duplicates Fattened)",
    xaxis_title="Step",
    yaxis_title="Event Type",
    yaxis=dict(type="category", categoryorder="array",
               categoryarray=["EVALUATE", "HINT", "GIVEUP"]),
    showlegend=False
)
fig.show()


In [7]:
import networkx as nx
import plotly.graph_objects as go
from collections import defaultdict



# --- Count transitions (step, from_event -> to_event) ---
transition_counts = defaultdict(int)
for flow in event_flows.values():
    for i in range(len(flow)-1):
        transition_counts[(i, flow[i], flow[i+1])] += 1

# --- Build graph ---
G = nx.DiGraph()
for (step, start, end), count in transition_counts.items():
    start_node = f"{start}_{step}"
    end_node = f"{end}_{step+1}"
    G.add_node(start_node, label=start, step=step)
    G.add_node(end_node, label=end, step=step+1)
    G.add_edge(start_node, end_node, weight=count)

# --- Layout ---
pos = {}
x_gap = 2
y_gap = 1
events_by_step = defaultdict(list)
for n, d in G.nodes(data=True):
    events_by_step[d['step']].append((n, d['label']))
for step, nodes in events_by_step.items():
    n = len(nodes)
    for i, (nname, label) in enumerate(sorted(nodes, key=lambda x:x[1])):
        pos[nname] = (step * x_gap, i * y_gap)

# --- Prepare edges and labels ---
edge_x, edge_y = [], []
edge_text_x, edge_text_y, edge_text = [], [], []

for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    # midpoint for label
    mx, my = (x0 + x1)/2, (y0 + y1)/2
    edge_text_x.append(mx)
    edge_text_y.append(my)
    edge_text.append(str(d['weight']))

# --- Edge trace ---
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=2, color='gray'),
    hoverinfo='none',
    mode='lines'
)

# --- Edge labels ---
edge_labels_trace = go.Scatter(
    x=edge_text_x, y=edge_text_y,
    text=edge_text,
    mode='text',
    textposition='middle center',
    hoverinfo='none'
)

# --- Nodes ---
node_x, node_y, node_text = [], [], []
for n, d in G.nodes(data=True):
    x, y = pos[n]
    node_x.append(x)
    node_y.append(y)
    node_text.append(f"{d['label']} (step {d['step']})")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=[d['label'] for n,d in G.nodes(data=True)],
    textposition='top center',
    marker=dict(size=20, color='lightblue', line_width=2),
    hovertext=node_text,
    hoverinfo='text'
)

# --- Plot ---
fig = go.Figure(data=[edge_trace, edge_labels_trace, node_trace])
fig.update_layout(
    title="Event Flow Graph with Edge Counts",
    showlegend=False,
    margin=dict(l=20, r=20, t=40, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)
fig.show()


In [8]:
import networkx as nx
import plotly.graph_objects as go
from collections import defaultdict


# --- Count transitions including self-loops ---
transition_counts = defaultdict(int)
for flow in event_flows.values():
    for i in range(len(flow)-1):
        start, end = flow[i], flow[i+1]
        transition_counts[(start, end)] += 1

# --- Build graph ---
G = nx.DiGraph()
for (start, end), count in transition_counts.items():
    G.add_node(start)
    G.add_node(end)
    G.add_edge(start, end, weight=count)

# --- Layout: circular for clarity ---
pos = nx.circular_layout(G)

# --- Prepare edges for Plotly ---
edge_x, edge_y = [], []
edge_text_x, edge_text_y, edge_text = [], [], []

for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    
    if u == v:  # self-loop: draw a small curve
        offset = 0.1
        edge_x += [x0, x0+offset, x0, None]
        edge_y += [y0, y0+offset, y0, None]
        edge_text_x.append(x0 + offset)
        edge_text_y.append(y0 + offset)
    else:
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_text_x.append((x0+x1)/2)
        edge_text_y.append((y0+y1)/2)
    
    edge_text.append(str(d['weight']))

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=2, color='gray'),
    hoverinfo='none',
    mode='lines'
)

# --- Edge labels ---
edge_labels_trace = go.Scatter(
    x=edge_text_x, y=edge_text_y,
    text=edge_text,
    mode='text',
    textposition='middle center',
    hoverinfo='none'
)

# --- Nodes ---
node_x, node_y, node_text = [], [], []
for n in G.nodes():
    x, y = pos[n]
    node_x.append(x)
    node_y.append(y)
    node_text.append(n)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=[n for n in G.nodes()],
    textposition='top center',
    marker=dict(size=30, color='lightblue', line_width=2),
    hovertext=node_text,
    hoverinfo='text'
)

# --- Plot ---
fig = go.Figure(data=[edge_trace, edge_labels_trace, node_trace])
fig.update_layout(
    title="Event Transition Graph (Including Self-Loops)",
    showlegend=False,
    margin=dict(l=20, r=20, t=40, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)
fig.show()


In [9]:
import networkx as nx
import plotly.graph_objects as go
from collections import defaultdict
import itertools
import numpy as np

# --- Count transitions ---
transition_counts = defaultdict(int)
for flow in event_flows.values():
    for i in range(len(flow)-1):
        transition_counts[(flow[i], flow[i+1])] += 1

# --- Nodes ---
all_events = list(set(itertools.chain.from_iterable(event_flows.values())))
Gtmp = nx.Graph()
Gtmp.add_nodes_from(all_events)
pos = nx.circular_layout(Gtmp)

# --- Helper for curved edges ---
def curved_edge(x0, y0, x1, y1, curvature=0.2):
    xm, ym = (x0+x1)/2, (y0+y1)/2
    dx, dy = x1-x0, y1-y0
    x_offset = -dy * curvature
    y_offset = dx * curvature
    return [x0, xm + x_offset, x1], [y0, ym + y_offset, y1]

# --- Build figure ---
fig = go.Figure()

for (start, end), count in transition_counts.items():
    x0, y0 = pos[start]
    x1, y1 = pos[end]

    if start == end:
        # small curve for self-loop
        offset = 0.1
        xs = [x0, x0+offset, x0]
        ys = [y0, y0+offset, y0]
        text_x, text_y = x0+offset, y0+offset
    else:
        xs, ys = curved_edge(x0, y0, x1, y1, curvature=0.2)
        text_x, text_y = xs[1], ys[1]

    # curved line
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        line=dict(width=2, color='gray'),
        mode='lines',
        hoverinfo='skip'
    ))

    # add a triangle at the end to mimic arrow
    arr_size = 0.05  # adjust for node distance
    angle = np.arctan2(y1 - ys[-2], x1 - xs[-2])
    arr_x = [x1 - arr_size*np.cos(angle+0.3), x1, x1 - arr_size*np.cos(angle-0.3)]
    arr_y = [y1 - arr_size*np.sin(angle+0.3), y1, y1 - arr_size*np.sin(angle-0.3)]
    fig.add_trace(go.Scatter(
        x=arr_x, y=arr_y, fill='toself', mode='lines', line=dict(color='black'),
        hoverinfo='skip', showlegend=False
    ))

    # count label
    fig.add_trace(go.Scatter(
        x=[text_x], y=[text_y],
        text=[str(count)],
        mode='text',
        textposition='middle center',
        showlegend=False,
        hoverinfo='skip'
    ))

# --- Nodes ---
node_x, node_y = [], []
for n in all_events:
    x, y = pos[n]
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=all_events,
    textposition='top center',
    marker=dict(size=30, color='lightblue', line_width=2),
    hovertext=all_events,
    hoverinfo='text',
    showlegend=False
)
fig.add_trace(node_trace)

# --- Layout ---
fig.update_layout(
    title="Event Transition Graph (Curved Arrows, Counts on Edge)",
    showlegend=False,
    margin=dict(l=20, r=20, t=40, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()


In [10]:
import networkx as nx
import plotly.graph_objects as go
from collections import defaultdict
import itertools
import numpy as np

# --- Count transitions ---
transition_counts = defaultdict(int)
for flow in event_flows.values():
    # normal transitions
    for i in range(len(flow)-1):
        transition_counts[(flow[i], flow[i+1])] += 1
    # start node
    transition_counts[("START", flow[0])] += 1
    # termination node
    transition_counts[(flow[-1], "TERMINATION")] += 1

# --- Nodes ---
all_events = list(set(itertools.chain.from_iterable(event_flows.values())))
all_events.extend(["START", "TERMINATION"])
Gtmp = nx.Graph()
Gtmp.add_nodes_from(all_events)
pos = nx.circular_layout(Gtmp)

# adjust positions of START and TERMINATION for clarity
pos["START"] = np.array([0, 1.2])
pos["TERMINATION"] = np.array([0, -1.2])

# --- Helper for curved edges ---
def curved_edge(x0, y0, x1, y1, curvature=0.2):
    xm, ym = (x0+x1)/2, (y0+y1)/2
    dx, dy = x1-x0, y1-y0
    x_offset = -dy * curvature
    y_offset = dx * curvature
    return [x0, xm + x_offset, x1], [y0, ym + y_offset, y1]

# --- Build figure ---
fig = go.Figure()

for (start, end), count in transition_counts.items():
    x0, y0 = pos[start]
    x1, y1 = pos[end]

    if start == end:
        offset = 0.1
        xs = [x0, x0+offset, x0]
        ys = [y0, y0+offset, y0]
        text_x, text_y = x0+offset, y0+offset
    else:
        xs, ys = curved_edge(x0, y0, x1, y1, curvature=0.2)
        text_x, text_y = xs[1], ys[1]

    # curved line
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        line=dict(width=2,color='gray'),
        mode='lines',
        hoverinfo='skip'
    ))

    # larger arrow
    arr_size = 0.07
    angle = np.arctan2(y1 - ys[-2], x1 - xs[-2])
    arr_x = [x1 - arr_size*np.cos(angle+0.3), x1, x1 - arr_size*np.cos(angle-0.3)]
    arr_y = [y1 - arr_size*np.sin(angle+0.3), y1, y1 - arr_size*np.sin(angle-0.3)]
    fig.add_trace(go.Scatter(
        x=arr_x, y=arr_y,
        fill='toself',
        mode='lines',
        line=dict(color='black'),
        hoverinfo='skip',
        showlegend=False
    ))

    # count label
    fig.add_trace(go.Scatter(
        x=[text_x], y=[text_y],
        text=[str(count)],
        mode='text',
        textposition='middle center',
        showlegend=False,
        hoverinfo='skip'
    ))

# --- Add nodes ---
node_x, node_y, node_color = [], [], []
for n in all_events:
    x, y = pos[n]
    node_x.append(x)
    node_y.append(y)
    if n == "START":
        node_color.append("green")
    elif n == "TERMINATION":
        node_color.append("red")
    else:
        node_color.append("lightblue")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=all_events,
    textposition='top center',
    marker=dict(size=30, color=node_color, line_width=2),
    hovertext=all_events,
    hoverinfo='text',
    showlegend=False
)
fig.add_trace(node_trace)

# --- Layout ---
fig.update_layout(
    title="Event Transition Graph with START (green) and TERMINATION (red)",
    showlegend=False,
    margin=dict(l=20, r=20, t=40, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()
